# Playwright MCP exploration notebook

This notebook is meant for **manual exploration** of the Clearfacts UI through the Playwright MCP browser wrapper.

Key idea: **keep one browser session open across cells**. You can log in once, then navigate step by step in later cells.

Typical flow:
1. Run setup and environment cells
2. Open the browser session
3. Log in as one role
4. Reuse the same `browser` object in later cells
5. Close the session when done


In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / "context_db").exists():
        root = root.parent
    if not (root / "context_db").exists():
        raise RuntimeError("Could not locate the cf-ml-context-db repository root from the current notebook session.")
    return root


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from context_db.agents.clearfacts_navigation_agent.schemas import ClearfactsNavigationTarget, ClearfactsRole
from context_db.agents.clearfacts_navigation_agent.tools import PlaywrightMcpBrowser, load_clearfacts_navigation_config

pd.set_option("display.max_colwidth", 160)
REPO_ROOT


PosixPath('/Users/dirkvangheel/Documents/dev/projects/clearfacts/cf-ml-context-db')

## Runtime configuration

This notebook defaults to a **visible Chrome session** with a small delay so you can follow along.

If you change these values after opening the browser, close and reopen the session.

In [2]:
os.environ["CLEARFACTS_TEST_USERS_FILE"] = str(
    (REPO_ROOT.parent / "cf-integration-tests" / "cypress" / "fixtures" / "users.json").resolve()
)
os.environ["PLAYWRIGHT_MCP_COMMAND"] = "npx"
os.environ["PLAYWRIGHT_MCP_ARGS"] = "-y @playwright/mcp@latest --browser chrome"
os.environ["PLAYWRIGHT_MCP_HEADLESS"] = "false"
os.environ["CLEARFACTS_NAVIGATION_STEP_DELAY_MS"] = "800"

runtime_settings = {
    key: os.environ[key]
    for key in [
        "CLEARFACTS_TEST_USERS_FILE",
        "PLAYWRIGHT_MCP_COMMAND",
        "PLAYWRIGHT_MCP_ARGS",
        "PLAYWRIGHT_MCP_HEADLESS",
        "CLEARFACTS_NAVIGATION_STEP_DELAY_MS",
    ]
}
display(pd.DataFrame([runtime_settings]))


,CLEARFACTS_TEST_USERS_FILE,PLAYWRIGHT_MCP_COMMAND,PLAYWRIGHT_MCP_ARGS,PLAYWRIGHT_MCP_HEADLESS,CLEARFACTS_NAVIGATION_STEP_DELAY_MS
0,/Users/dirkvangheel/Documents/dev/projects/clearfacts/cf-integration-tests/cypress/fixtures/users.json,npx,-y @playwright/mcp@latest --browser chrome,false,800


In [3]:
config = load_clearfacts_navigation_config()
browser = None


def display_call(call) -> None:
    lines = [
        f"### Tool call: `{call.tool_name}`",
        "**Arguments**",
        f"```json\n{json.dumps(call.arguments, indent=2, ensure_ascii=True)}\n```",
    ]
    if call.message:
        lines.extend([
            "**Message**",
            f"```\n{call.message[:4000]}\n```",
        ])
    display(Markdown("\n".join(lines)))


def display_page(page) -> None:
    if page is None:
        display(Markdown("_No page evidence available._"))
        return

    lines = [
        "### Page summary",
        f"- **URL:** {page.url or '_unknown_'}",
        f"- **Title:** {page.title or '_unknown_'}",
    ]
    if page.text_excerpt:
        lines.extend([
            "- **Visible text excerpt:**",
            f"```\n{page.text_excerpt[:4000]}\n```",
        ])
    display(Markdown("\n".join(lines)))
    if page.snapshot:
        display(Markdown("### Snapshot excerpt"))
        display(Markdown(f"```\n{page.snapshot[:4000]}\n```"))


async def ensure_browser_open() -> PlaywrightMcpBrowser:
    global browser
    if browser is None:
        browser = PlaywrightMcpBrowser(config.playwright)
        await browser.__aenter__()
    return browser


async def close_browser() -> None:
    global browser
    if browser is not None:
        await browser.__aexit__(None, None, None)
        browser = None


async def login_as(role: ClearfactsRole = ClearfactsRole.SME_ADMIN):
    active_browser = await ensure_browser_open()
    credentials = config.roles[role]
    landing = config.landing_expectations.get(role)

    calls = []
    calls.append(await active_browser.navigate(f"{config.base_url}/login"))
    calls.append(await active_browser.fill("#username", credentials.username))
    calls.append(await active_browser.fill("#password", credentials.password))
    calls.append(await active_browser.click("#_submit"))

    if landing is not None:
        if landing.text:
            calls.append(await active_browser.wait_for_text(landing.text))
        elif landing.selector:
            calls.append(await active_browser.wait_for_selector(landing.selector))

    page = await active_browser.inspect_page(include_snapshot=True)
    return calls, page


async def open_target(target: ClearfactsNavigationTarget):
    active_browser = await ensure_browser_open()
    target_definition = config.navigation_targets[target]
    calls = []

    if target in {ClearfactsNavigationTarget.AUTOMATION_PURCHASE, ClearfactsNavigationTarget.AUTOMATION_SALE}:
        calls.append(await active_browser.click('[data-testid="sidenav-automation"] > span'))

    calls.append(await active_browser.click(target_definition.selector))
    if target_definition.expected_text:
        calls.append(await active_browser.wait_for_text(target_definition.expected_text))

    page = await active_browser.inspect_page(include_snapshot=True)
    return calls, page


async def inspect_current_page():
    active_browser = await ensure_browser_open()
    return await active_browser.inspect_page(include_snapshot=True)


async def logout():
    active_browser = await ensure_browser_open()
    call = await active_browser.navigate(f"{config.base_url}/logout")
    await active_browser.wait_for_text("Aanmelden op Clearfacts")
    page = await active_browser.inspect_page(include_snapshot=True)
    return [call], page


## 1. Open one persistent browser session

Run this cell once. The returned `browser` object stays available for the later cells.

In [4]:
active_browser = await ensure_browser_open()
display(Markdown("## Available tools\n\n" + "\n".join(f"- `{tool}`" for tool in active_browser.tool_inventory())))


## Available tools

- `browser_click`
- `browser_close`
- `browser_console_messages`
- `browser_drag`
- `browser_drop`
- `browser_evaluate`
- `browser_file_upload`
- `browser_fill_form`
- `browser_handle_dialog`
- `browser_hover`
- `browser_navigate`
- `browser_navigate_back`
- `browser_network_requests`
- `browser_press_key`
- `browser_resize`
- `browser_run_code`
- `browser_select_option`
- `browser_snapshot`
- `browser_tabs`
- `browser_take_screenshot`
- `browser_type`
- `browser_wait_for`

## 2. Log in as SME admin

In [7]:
config.navigation_targets.keys()

dict_keys([<ClearfactsNavigationTarget.PURCHASE: 'purchase'>, <ClearfactsNavigationTarget.SALE: 'sale'>, <ClearfactsNavigationTarget.VARIOUS: 'various'>, <ClearfactsNavigationTarget.WORKFLOW: 'workflow'>, <ClearfactsNavigationTarget.AUTOMATION_PURCHASE: 'automation_purchase'>, <ClearfactsNavigationTarget.AUTOMATION_SALE: 'automation_sale'>])

In [5]:
calls, page = await login_as(ClearfactsRole.SME_ADMIN)
for call in calls:
    display_call(call)
display_page(page)


NavigationExecutionError: Timed out while waiting for selector '#page-dashboard'.

## 3. Open the purchase inbox

In [ ]:
calls, page = await open_target(ClearfactsNavigationTarget.PURCHASE)
for call in calls:
    display_call(call)
display_page(page)


## 4. Quick page inspection

Useful after any manual navigation step.

In [ ]:
page = await inspect_current_page()
display_page(page)


## 5. Explore other inboxes

These are simple, safe follow-up actions using the same logged-in session.

In [ ]:
calls, page = await open_target(ClearfactsNavigationTarget.SALE)
for call in calls:
    display_call(call)
display_page(page)


In [ ]:
calls, page = await open_target(ClearfactsNavigationTarget.VARIOUS)
for call in calls:
    display_call(call)
display_page(page)


## 6. Direct low-level Playwright wrapper actions

This is useful to get a feel for the lower-level operations without going through the agent planner.

In [ ]:
active_browser = await ensure_browser_open()
call = await active_browser.click('[data-node-target="purchase"]')
display_call(call)
page = await active_browser.inspect_page(include_snapshot=True)
display_page(page)


In [ ]:
call = await active_browser.wait_for_text("Digitale postbus: Aankoop")
display_call(call)
page = await active_browser.inspect_page(include_snapshot=True)
display_page(page)


## 7. Optional: logout while keeping control in the notebook

In [ ]:
calls, page = await logout()
for call in calls:
    display_call(call)
display_page(page)


## 8. Close the persistent browser session

Run this when you are done, or before changing the Playwright environment variables.

In [ ]:
await close_browser()
browser
